# 03 — KArSL Keypoints Verification

Read-only verification of precomputed Kaggle keypoint subset and recorded full structural QC. No MediaPipe extraction, and no files are overwritten. Source license remains unresolved.

## Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Keypoint subset, splits and structural audit

In [ ]:
from pathlib import Path
import pandas as pd
import os
B = Path(os.environ.get(
    "SIGNOVA_KARSL_ROOT",
    str(Path("/content") / "drive" / "MyDrive" / "SIGNOVA" / "KArSL")
))
R = B / "reports"
k = pd.read_csv(R / "keypoints_training_manifest.csv", low_memory=False)
s = pd.read_csv(R / "keypoints_split_manifest.csv", low_memory=False)
q = pd.read_csv(R / "keypoints_quality_full.csv", low_memory=False)
ids = set(k["sample_id"].astype(str))
checks = {
    "Unique keypoint samples": k["sample_id"].is_unique,
    "Three signers": k["signer_id"].nunique() == 3,
    "No duplicate sample per fold": not s.duplicated(["fold", "sample_id"]).any(),
    "Three folds": s["fold"].nunique() == 3,
    "All folds cover the subset": all(set(g["sample_id"].astype(str)) == ids for _,g in s.groupby("fold")),
    "No signer leakage": all(g.groupby("signer_id")["split"].nunique().max() == 1 for _,g in s.groupby("fold")),
    "Quality report covers subset": set(q["sample_id"].astype(str)) == ids,
    "Recorded structural checks pass": q["status"].eq("PASS").all()
}
print("=== KEYPOINT BENCHMARK ===")
print("Samples:", len(k))
print("Classes:", k["class_label"].nunique())
print("Signers:", k["signer_id"].nunique())
print("Samples with zero groups:", q["zero_group_count"].gt(0).sum())
print("=== VALIDATION ===")
for name, result in checks.items():
    print(name, "PASS" if result else "FAIL")
assert all(checks.values()), "Keypoint benchmark checks failed"

## On-disk keypoints spot check

In [ ]:
from pathlib import Path
import pandas as pd
import os
B = Path(os.environ.get(
    "SIGNOVA_KARSL_ROOT",
    str(Path("/content") / "drive" / "MyDrive" / "SIGNOVA" / "KArSL")
))
R = B / "reports"
import numpy as np
k = pd.read_csv(R / "keypoints_training_manifest.csv", low_memory=False)
expected = {"lh": 63, "pose": 99, "rh": 63}
checked = 0
issues = []
for row in k.sample(min(20, len(k)), random_state=42).itertuples(index=False):
    lengths = []
    for group, width in expected.items():
        path = Path(getattr(row, f"{group}_path"))
        try:
            a = np.load(path, mmap_mode="r", allow_pickle=False)
            if a.ndim != 2 or a.shape[0] == 0 or a.shape[1] != width or not np.isfinite(a).all():
                issues.append((row.sample_id, group, "INVALID_DATA"))
            else:
                lengths.append(a.shape[0])
            checked += 1
        except Exception as e:
            issues.append((row.sample_id, group, type(e).__name__))
    if len(lengths) == 3 and len(set(lengths)) != 1:
        issues.append((row.sample_id, "all", "FRAME_MISMATCH"))
print("=== ON-DISK SPOT CHECK ===")
print("Files checked:", checked)
print("Issues:", len(issues))
print("Issue examples:", issues[:5])
print("Full structural audit already recorded: YES")
print("Zero-group semantic review: PENDING")
print("Kaggle dataset usage rights: UNVERIFIED")
print("Original keypoints modified: NO")

In [ ]:
from pathlib import Path
import shutil
import json

D = Path("/content/drive/MyDrive")
source = D / "Colab Notebooks/SIGNOVA_KArSL_Clean"
target = D / "SIGNOVA/KArSL/github_upload/karsl/notebooks"

names = [
    "01_KArSL_Inventory.ipynb",
    "02_KArSL_QC_and_Splits.ipynb",
    "03_KArSL_Keypoints_Audit.ipynb"
]

target.mkdir(parents=True, exist_ok=True)

print("=== NOTEBOOK SOURCE CHECK ===")
for name in names:
    print(name, "FOUND" if (source / name).is_file() else "MISSING")

if not all((source / name).is_file() for name in names):
    print("STATUS: STOPPED")
else:
    for old in target.glob("*.ipynb"):
        old.unlink()

    for name in names:
        nb = json.loads((source / name).read_text(encoding="utf-8"))

        for cell in nb.get("cells", []):
            if cell.get("cell_type") == "code":
                cell["outputs"] = []
                cell["execution_count"] = None

        (target / name).write_text(
            json.dumps(nb, ensure_ascii=False, indent=1),
            encoding="utf-8"
        )

    print("\n=== STAGED NOTEBOOKS ===")
    for p in sorted(target.glob("*.ipynb")):
        print(p.name, "READY")

    print("\n=== FINAL STATUS ===")
    print("Staged notebooks:", len(list(target.glob("*.ipynb"))))
    print("Original notebooks modified: NO")
    print("GitHub upload: NOT STARTED")

In [ ]:
from pathlib import Path
import json

D = Path("/content/drive/MyDrive")
T = D / "SIGNOVA/KArSL/github_upload/karsl/notebooks"

names = [
    "01_KArSL_Inventory.ipynb",
    "02_KArSL_QC_and_Splits.ipynb",
    "03_KArSL_Keypoints_Audit.ipynb"
]

found = {}

print("=== NOTEBOOK DISCOVERY ===")

for name in names:
    matches = [
        p for p in D.rglob(name)
        if "github_upload" not in p.parts
    ]
    print(name, "FOUND" if matches else "MISSING")

    for p in matches:
        print("Source:", p.relative_to(D))

    if matches:
        found[name] = max(matches, key=lambda p: p.stat().st_mtime)

if len(found) != 3:
    print("\nSTATUS: STOPPED - Missing notebooks")
else:
    T.mkdir(parents=True, exist_ok=True)

    for name, src in found.items():
        nb = json.loads(src.read_text(encoding="utf-8"))

        for cell in nb.get("cells", []):
            if cell.get("cell_type") == "code":
                cell["outputs"] = []
                cell["execution_count"] = None

        (T / name).write_text(
            json.dumps(nb, ensure_ascii=False, indent=1),
            encoding="utf-8"
        )

    print("\n=== STAGED NOTEBOOKS ===")
    for name in names:
        print(name, "READY")

    print("\n=== FINAL STATUS ===")
    print("Notebooks staged:", len(found))
    print("Original notebooks modified: NO")
    print("Other staged notebooks preserved: YES")
    print("GitHub upload: NOT STARTED")